# Build the manuscript to Word

Converts `Manuscript/Draft/manuscript.md` to `Manuscript/Draft/manuscript.docx`.

**Why this notebook exists.** Until 8 August 2026 the `.docx` was produced by some path
outside the repository and nothing here could rebuild it. That made the final deliverable the
one artefact in the project that was not reproducible, in a study whose subject is
reproducibility. This notebook closes that gap.

**Run it after any edit to `manuscript.md`.** It is the last step in the chain and reads
nothing but the markdown and the forest plot, so it never needs the tracker or the exports.

**Scope.** It handles exactly the markdown this manuscript uses and nothing more. Headings at
three levels, paragraphs wrapped across lines, pipe tables, horizontal rules, one block
quotation, one embedded figure, and inline bold and inline code. It deliberately does not
implement ordered lists, because several wrapped prose lines in the reference list begin with a
year followed by a full stop and a list parser would silently eat them.

If you add a markdown construct the manuscript does not currently use, the strict check in the
last cell will fail rather than dropping it quietly.

In [1]:
import re
from pathlib import Path

import docx
from docx import Document
from docx.enum.section import WD_SECTION
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
SRC = ROOT / "Manuscript" / "Draft" / "manuscript.md"
OUT = ROOT / "Manuscript" / "Draft" / "manuscript.docx"
FIGDIR = ROOT / "Drafts" / "figures"

print("python-docx", docx.__version__)
print("source     ", SRC, SRC.exists())
print("target     ", OUT)

python-docx 1.2.0
source      C:\Users\adamt\Desktop\Capstone - AI Imaging Appraisal\Manuscript\Draft\manuscript.md True
target      C:\Users\adamt\Desktop\Capstone - AI Imaging Appraisal\Manuscript\Draft\manuscript.docx


## Parse

The markdown is turned into a flat list of blocks first, so the parse can be inspected before
anything is written. Every source line lands in exactly one block or is an intentional blank,
and the audit cell at the end checks that.

In [2]:
RE_HEADING = re.compile(r"^(#{1,6})\s+(.*)$")
RE_IMAGE = re.compile(r"^!\[(?P<alt>[^\]]*)\]\((?P<path>[^)]+)\)\s*$")
RE_TABLE_SEP = re.compile(r"^\|[\s:|-]+\|$")


def split_row(line):
    """Split a pipe table row, dropping the empty edges either side."""
    return [c.strip() for c in line.strip().strip("|").split("|")]


def parse_blocks(text):
    lines = text.split("\n")
    blocks, i, consumed = [], 0, 0

    while i < len(lines):
        raw = lines[i]
        line = raw.rstrip()

        if not line.strip():
            i += 1
            continue

        if line.strip() == "---":
            blocks.append({"kind": "rule"})
            i += 1
            consumed += 1
            continue

        m = RE_IMAGE.match(line)
        if m:
            blocks.append({"kind": "image", "alt": m.group("alt"), "path": m.group("path")})
            i += 1
            consumed += 1
            continue

        m = RE_HEADING.match(line)
        if m:
            blocks.append({"kind": "heading", "level": len(m.group(1)), "text": m.group(2).strip()})
            i += 1
            consumed += 1
            continue

        if line.startswith("|"):
            start = i
            rows = []
            while i < len(lines) and lines[i].strip().startswith("|"):
                rows.append(lines[i].rstrip())
                i += 1
            consumed += i - start
            body = [r for r in rows if not RE_TABLE_SEP.match(r.strip())]
            has_header = len(body) < len(rows)
            blocks.append(
                {
                    "kind": "table",
                    "rows": [split_row(r) for r in body],
                    "has_header": has_header,
                }
            )
            continue

        if line.startswith(">"):
            start = i
            parts = []
            while i < len(lines) and lines[i].startswith(">"):
                parts.append(lines[i].lstrip(">").strip())
                i += 1
            consumed += i - start
            blocks.append({"kind": "quote", "text": " ".join(p for p in parts if p)})
            continue

        # Otherwise a paragraph, running until a blank line or the start of another construct.
        start = i
        parts = []
        while i < len(lines):
            nxt = lines[i].rstrip()
            if not nxt.strip() or nxt.strip() == "---":
                break
            if nxt.startswith("|") or nxt.startswith(">"):
                break
            if RE_HEADING.match(nxt) or RE_IMAGE.match(nxt):
                break
            parts.append(nxt.strip())
            i += 1
        consumed += i - start
        blocks.append({"kind": "para", "text": " ".join(parts)})

    return blocks, consumed


text = SRC.read_text(encoding="utf-8")
blocks, consumed = parse_blocks(text)

counts = {}
for b in blocks:
    counts[b["kind"]] = counts.get(b["kind"], 0) + 1
print("blocks parsed:", counts)
print("total blocks :", len(blocks))

blocks parsed: {'heading': 58, 'para': 223, 'rule': 30, 'table': 14, 'quote': 1, 'image': 1}
total blocks : 327


## Styling

A body font, heading sizes, and a table style that puts a rule under the header row rather than
boxing every cell. The document is set to A4 with margins wide enough for a printed submission.

In [3]:
BODY_FONT = "Calibri"
BODY_SIZE = Pt(11)
MONO_FONT = "Consolas"

RE_INLINE = re.compile(r"(\*\*.+?\*\*|`[^`]+`)", re.S)


def add_runs(paragraph, s, *, bold=False, size=None, mono_ok=True):
    """Write `s` into `paragraph`, honouring **bold** and `code` spans."""
    for piece in RE_INLINE.split(s):
        if not piece:
            continue
        if piece.startswith("**") and piece.endswith("**") and len(piece) > 4:
            run = paragraph.add_run(piece[2:-2])
            run.bold = True
        elif mono_ok and piece.startswith("`") and piece.endswith("`") and len(piece) > 2:
            run = paragraph.add_run(piece[1:-1])
            run.font.name = MONO_FONT
            run.font.color.rgb = RGBColor(0x33, 0x33, 0x33)
        else:
            run = paragraph.add_run(piece)
        if bold:
            run.bold = True
        if size is not None:
            run.font.size = size
    return paragraph


def set_cell_border_bottom(cell, sz=8):
    tcPr = cell._tc.get_or_add_tcPr()
    borders = OxmlElement("w:tcBorders")
    bottom = OxmlElement("w:bottom")
    bottom.set(qn("w:val"), "single")
    bottom.set(qn("w:sz"), str(sz))
    bottom.set(qn("w:color"), "000000")
    borders.append(bottom)
    tcPr.append(borders)


def add_horizontal_rule(doc):
    p = doc.add_paragraph()
    pPr = p._p.get_or_add_pPr()
    bdr = OxmlElement("w:pBdr")
    bottom = OxmlElement("w:bottom")
    bottom.set(qn("w:val"), "single")
    bottom.set(qn("w:sz"), "6")
    bottom.set(qn("w:color"), "BBBBBB")
    bdr.append(bottom)
    pPr.append(bdr)
    p.paragraph_format.space_before = Pt(6)
    p.paragraph_format.space_after = Pt(10)
    return p


def new_document(compact=False):
    doc = Document()
    normal = doc.styles["Normal"]
    normal.font.name = BODY_FONT
    normal.font.size = Pt(10) if compact else BODY_SIZE
    normal.paragraph_format.space_after = Pt(4) if compact else Pt(8)
    normal.paragraph_format.line_spacing = 1.0 if compact else 1.15

    for name, size, colour in [
        ("Heading 1", Pt(20), RGBColor(0, 0, 0)),
        ("Heading 2", Pt(14), RGBColor(0, 0, 0)),
        ("Heading 3", Pt(12), RGBColor(0x33, 0x33, 0x33)),
    ]:
        st = doc.styles[name]
        st.font.name = BODY_FONT
        st.font.size = size
        st.font.bold = True
        st.font.color.rgb = colour

    sec = doc.sections[0]
    sec.page_width, sec.page_height = Inches(8.27), Inches(11.69)
    m = Inches(0.75) if compact else Inches(1.0)
    sec.left_margin = sec.right_margin = m
    sec.top_margin = sec.bottom_margin = m
    return doc


print("styling helpers ready")

styling helpers ready


## Render

The title block is handled specially. Everything after it is written in document order.

In [4]:
def render(blocks, doc):
    figures = 0
    tables = 0

    for idx, b in enumerate(blocks):
        kind = b["kind"]

        if kind == "heading":
            level = b["level"]
            p = doc.add_paragraph(style=f"Heading {min(level, 3)}")
            add_runs(p, b["text"])
            if level == 1:
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
                p.paragraph_format.space_after = Pt(4)
            elif level == 3 and idx < 4:
                # The subtitle immediately under the title.
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            else:
                p.paragraph_format.space_before = Pt(14)
                p.paragraph_format.space_after = Pt(4)
                p.paragraph_format.keep_with_next = True

        elif kind == "para":
            p = doc.add_paragraph()
            add_runs(p, b["text"])
            # Table and figure captions sit tight against what they label.
            if re.match(r"^\*\*(Table|Figure) \d+\.\*\*", b["text"]):
                p.paragraph_format.space_before = Pt(10)
                p.paragraph_format.space_after = Pt(4)
                p.paragraph_format.keep_with_next = True
            if idx < 6 and not b["text"].startswith("**"):
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER

        elif kind == "quote":
            p = doc.add_paragraph()
            add_runs(p, b["text"])
            p.paragraph_format.left_indent = Inches(0.4)
            p.paragraph_format.right_indent = Inches(0.4)
            p.paragraph_format.space_before = Pt(6)
            p.paragraph_format.space_after = Pt(10)
            for run in p.runs:
                run.italic = True

        elif kind == "rule":
            add_horizontal_rule(doc)

        elif kind == "image":
            src = (SRC.parent / b["path"]).resolve()
            if not src.exists():
                raise FileNotFoundError(f"figure not found: {src}")
            p = doc.add_paragraph()
            p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            p.add_run().add_picture(str(src), width=Inches(6.0))
            p.paragraph_format.space_before = Pt(8)
            p.paragraph_format.space_after = Pt(4)
            p.paragraph_format.keep_with_next = True
            figures += 1

        elif kind == "table":
            rows = b["rows"]
            ncols = max(len(r) for r in rows)
            t = doc.add_table(rows=0, cols=ncols)
            t.style = "Table Grid" if not b["has_header"] else "Table Grid"
            t.alignment = WD_TABLE_ALIGNMENT.CENTER
            for ri, row in enumerate(rows):
                cells = t.add_row().cells
                for ci in range(ncols):
                    val = row[ci] if ci < len(row) else ""
                    cell = cells[ci]
                    cell.text = ""
                    para = cell.paragraphs[0]
                    para.paragraph_format.space_after = Pt(2)
                    para.paragraph_format.space_before = Pt(2)
                    is_head = b["has_header"] and ri == 0
                    add_runs(para, val, bold=is_head, size=Pt(10))
            tables += 1

        else:
            raise ValueError(f"unhandled block kind: {kind}")

    return figures, tables


doc = new_document()
n_fig, n_tab = render(blocks, doc)
doc.save(OUT)
print(f"wrote {OUT}")
print(f"  figures embedded: {n_fig}")
print(f"  tables written  : {n_tab}")
print(f"  size            : {OUT.stat().st_size:,} bytes")

wrote C:\Users\adamt\Desktop\Capstone - AI Imaging Appraisal\Manuscript\Draft\manuscript.docx
  figures embedded: 1
  tables written  : 14
  size            : 217,795 bytes


## Audit the build

A silent conversion that drops content is worse than one that fails. These checks compare the
built document back against the source and raise if anything went missing.

In [5]:
check = Document(OUT)

src_headings = [b for b in blocks if b["kind"] == "heading"]
src_tables = [b for b in blocks if b["kind"] == "table"]
src_captions = re.findall(r"^\*\*(Table|Figure) (\d+)\.\*\*", text, re.M)

doc_headings = [p for p in check.paragraphs if p.style.name.startswith("Heading")]
doc_images = check.inline_shapes

problems = []

if len(doc_headings) != len(src_headings):
    problems.append(f"headings {len(doc_headings)} in docx vs {len(src_headings)} in markdown")
if len(check.tables) != len(src_tables):
    problems.append(f"tables {len(check.tables)} in docx vs {len(src_tables)} in markdown")
if len(doc_images) != sum(1 for b in blocks if b["kind"] == "image"):
    problems.append(f"images {len(doc_images)} in docx vs {sum(1 for b in blocks if b['kind']=='image')}")

# Every source line must have been consumed by exactly one block or been blank.
nonblank = sum(1 for l in text.split("\n") if l.strip())
if consumed != nonblank:
    problems.append(f"consumed {consumed} source lines but {nonblank} are non blank")

# Caption numbering must be a clean run for tables and for figures separately.
tbl_nums = [int(n) for k, n in src_captions if k == "Table"]
fig_nums = [int(n) for k, n in src_captions if k == "Figure"]
if tbl_nums != list(range(1, len(tbl_nums) + 1)):
    problems.append(f"table caption numbering is not sequential: {tbl_nums}")
if fig_nums != list(range(1, len(fig_nums) + 1)):
    problems.append(f"figure caption numbering is not sequential: {fig_nums}")

# No unrendered markdown should survive into the body text.
body = "\n".join(p.text for p in check.paragraphs)
for pat, label in [(r"\*\*", "bold markers"), (r"(?<!\w)`(?!\w*`)", "stray backtick")]:
    if re.search(pat, body):
        problems.append(f"unrendered {label} left in the document body")

if "[Author name]" in body:
    problems.append("author placeholder still present")

print(f"headings : {len(doc_headings):>4}")
print(f"tables   : {len(check.tables):>4}")
print(f"figures  : {len(doc_images):>4}")
print(f"captions : {len(tbl_nums)} tables, {len(fig_nums)} figures")
print()
if problems:
    for p in problems:
        print("  FAIL ", p)
    raise AssertionError(f"{len(problems)} build check(s) failed")
print("all build checks passed")

headings :   58


tables   :   14
figures  :    1
captions : 14 tables, 1 figures

all build checks passed


## Build the capstone summary sheet

The same converter, pointed at the one to two page summary sheet that accompanies the conference
submission. It reuses the parser and the renderer defined above, so the two documents cannot drift
apart in styling.

The summary sheet has no figures, so the image branch is simply never taken. The checks below are
different from the manuscript's, because what matters here is that every field the capstone brief
requires is actually present and that the abstract sits inside the required word window.


In [6]:
SUMMARY_SRC = ROOT / "Manuscript" / "Submission" / "summary_sheet.md"
SUMMARY_OUT = ROOT / "Manuscript" / "Submission" / "summary_sheet.docx"

s_text = SUMMARY_SRC.read_text(encoding="utf-8")
s_blocks, s_consumed = parse_blocks(s_text)

s_doc = new_document(compact=True)   # the brief caps this at two pages
s_fig, s_tab = render(s_blocks, s_doc)
s_doc.save(SUMMARY_OUT)

s_check = Document(SUMMARY_OUT)
s_body = " ".join(p.text for p in s_check.paragraphs)
s_cells = " ".join(c.text for t in s_check.tables for r in t.rows for c in r.cells)
s_all = s_body + " " + s_cells

s_problems = []

s_nonblank = sum(1 for l in s_text.splitlines() if l.strip())
if s_consumed != s_nonblank:
    s_problems.append(f"consumed {s_consumed} of {s_nonblank} non blank source lines")

if len(s_check.tables) != sum(1 for b in s_blocks if b["kind"] == "table"):
    s_problems.append("table count mismatch")

if "**" in s_body:
    s_problems.append("unrendered bold markers in body")

# Every field the capstone brief requires must be present by name.
REQUIRED = [
    "Conference", "Symposium", "Conference site", "call for papers",
    "Submission deadline", "Author notification", "Event dates", "Event location",
    "Abstract", "Gemini 3.6 Flash",
]
for field in REQUIRED:
    if field.lower() not in s_all.lower():
        s_problems.append(f"required field missing: {field}")

# The abstract must sit inside the 100 to 250 word window the brief specifies.
abstract = s_text.split("## Abstract")[1].split(chr(10) + "## ")[0]
a_words = len(re.findall(r"[A-Za-z0-9][A-Za-z0-9'.-]*", abstract))
if not (100 <= a_words <= 250):
    s_problems.append(f"abstract is {a_words} words, outside the required 100 to 250")

# The paper and the summary sheet must quote the same abstract.
tex = (ROOT / "Manuscript" / "Submission" / "main.tex").read_text(encoding="utf-8")
tex_abs = tex.split("begin{abstract}")[1].split("end{abstract}")[0]
norm = lambda s: " ".join(s.replace("$p$", "p").replace(chr(92), " ").split())
if norm(tex_abs) != norm(abstract):
    s_problems.append("abstract in main.tex differs from the summary sheet")

print(f"wrote {SUMMARY_OUT}")
print(f"  tables   : {s_tab}")
print(f"  abstract : {a_words} words")
print(f"  size     : {SUMMARY_OUT.stat().st_size:,} bytes")
print()
if s_problems:
    for x in s_problems:
        print("  FAIL ", x)
    raise AssertionError(f"{len(s_problems)} summary sheet check(s) failed")
print("all summary sheet checks passed")


wrote C:\Users\adamt\Desktop\Capstone - AI Imaging Appraisal\Manuscript\Submission\summary_sheet.docx
  tables   : 1
  abstract : 209 words
  size     : 39,455 bytes

all summary sheet checks passed
